# Capítulo 2 — Divisão em pedaços (Text Chunking)

**Livro:** começa na página **94** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## Context window limits the maximum chunk length · PDF p. 96

#### 🧪 Exercício: Quantos tokens cabem?

Conte os tokens de cada documento da base com `contar_tokens` e o total. Se o limite de contexto fosse de 300 tokens, a base inteira caberia num único *prompt*?

In [ ]:
tok = {d["id"]: contar_tokens(d["texto"]) for d in DOCS}
print(tok, "total:", sum(tok.values()), "-> cabe" if sum(tok.values()) <= 300 else "-> não cabe")

## Different chunking strategies · PDF p. 101

#### 🧪 Exercício: Três estratégias de chunking

Junte todos os textos e divida de três jeitos: (a) tamanho fixo de 200 caracteres com sobreposição de 40; (b) por frases, agrupando até 200 caracteres; (c) recursivo, tentando separar por `\n\n`, depois `. `, depois espaço. Compare quantos pedaços cada um gera e se algum corta palavras.

In [ ]:
texto = "\n\n".join(d["texto"] for d in DOCS)
def fixo(t, tam=200, sobre=40): return [t[i:i + tam] for i in range(0, len(t), tam - sobre)]
def por_frases(t, tam=200):
    pedacos, atual = [], ""
    for f in re.split(r"(?<=[.!?])\s+", t):
        if len(atual) + len(f) > tam and atual: pedacos.append(atual.strip()); atual = ""
        atual += f + " "
    return pedacos + [atual.strip()] if atual.strip() else pedacos
def recursivo(t, tam=200, seps=("\n\n", ". ", " ")):
    if len(t) <= tam or not seps: return [t]
    partes, saida, atual = t.split(seps[0]), [], ""
    for p in partes:
        cand = (atual + seps[0] + p) if atual else p
        if len(cand) <= tam: atual = cand
        else:
            if atual: saida.append(atual)
            atual = p if len(p) <= tam else ""
            if len(p) > tam: saida += recursivo(p, tam, seps[1:])
    return saida + ([atual] if atual else [])
for nome, f in [("fixo", fixo), ("frases", por_frases), ("recursivo", recursivo)]:
    ps = f(texto); print(nome, len(ps), "pedaços | começa no meio de palavra:", sum(1 for p in ps[1:] if p[:1].isalnum() and p[:1].islower()))

## Advanced indexing techniques related to chunking · PDF p. 113

#### 🧪 Exercício: Indexar o resumo, devolver o documento

Para cada documento, peça ao modelo um resumo de uma frase. Indexe os **resumos** (busca por palavras), mas devolva o **texto completo** do documento encontrado (ideia de *multi-vector / summary index*). Teste com "posso levar meu cachorro?".

In [ ]:
resumos = {d["id"]: perguntar(f"Resuma em uma frase curta, em português:\n{d['texto']}") for d in DOCS}
def busca_palavras(q, textos):
    qs = set(re.findall(r"\w+", q.lower()))
    return max(textos, key=lambda k: len(qs & set(re.findall(r"\w+", textos[k].lower()))))
achado = busca_palavras("posso levar meu cachorro na biblioteca?", resumos)
print(resumos[achado]); print("->", next(d["texto"] for d in DOCS if d["id"] == achado))

## Workflow of HyDE technology · PDF p. 119

#### 🧪 Exercício: HyDE: buscar com uma resposta hipotética

Implemente HyDE: peça ao modelo uma resposta **hipotética** para "quanto pago se atrasar um livro?", gere o embedding dessa resposta e use para buscar o documento mais parecido. Compare com buscar pelo embedding da própria pergunta.

In [ ]:
q = "quanto pago se atrasar um livro?"
E_docs = embeddings([d["texto"] for d in DOCS])
hipotese = perguntar(f"Escreva um parágrafo curto que responderia: {q}")
for nome, v in [("pergunta", embeddings(q)[0]), ("HyDE", embeddings(hipotese)[0])]:
    s = similaridade(E_docs, v); i = int(np.argmax(s)); print(nome, DOCS[i]["titulo"], round(float(s[i]), 3))